<a href="https://colab.research.google.com/github/harshvarudkar/test/blob/master/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U langchain langchain-core langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.4 MB/s eta 0:00:00


In [5]:
import os
from getpass import getpass
from google.colab import userdata

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [7]:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [13]:
# Build prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a unit converter  assistant. Answer concisely."),
    ("human", "{question}")
])


In [14]:
# Create model
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2
)

In [15]:
# Create parser
parser = StrOutputParser()

In [16]:
# LCEL chain: prompt -> model -> parser
chain = prompt | llm | parser

In [17]:
# Run it
response = chain.invoke({"question": "What is 100 meter to inches?"})
print(response)

100 meters is approximately 3937.01 inches.


Multi-step LangChain Example

Step 1 structures messy notes. Step 2 produces a leadership-ready summary with actions and escalations.

In [18]:
!pip install -q -U langchain langchain-core langchain-openai

In [19]:
import os
from google.colab import userdata
from operator import itemgetter

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [20]:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [21]:
# Sample meeting notes
meeting_notes = """
Client steering committee meeting notes:
- Client is concerned about delays in data migration.
- Testing completion slipped by one week due to environment issues.
- Program manager asked for a revised milestone plan by Friday.
- Finance lead wants a clearer view of budget burn and forecast.
- Team agreed to prioritize critical interfaces before lower-value reports.
- There is a risk that user training may start late unless materials are approved this week.
- Executive sponsor asked for a concise status summary for next Monday.
"""

In [22]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
parser = StrOutputParser()

In [23]:
# Step 1: extract structured notes
extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a program management analyst."),
    ("human",
     "Review these meeting notes:\n\n{notes}\n\n"
     "Extract:\n"
     "1. Key decisions\n"
     "2. Risks/issues\n"
     "3. Action items\n"
     "4. Deadlines\n"
     "5. Stakeholders mentioned")
])

In [24]:
extract_chain = extract_prompt | llm | parser

In [25]:
# Step 2: write executive-ready summary
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a PMO lead writing for executives."),
    ("human",
     "Using this extracted meeting information:\n\n{extracted}\n\n"
     "Write:\n"
     "- Executive summary\n"
     "- Top priorities this week\n"
     "- Action items with owners\n"
     "- Any escalation points")
])

In [26]:
meeting_chain = (
    {
        "extracted": itemgetter("notes") | extract_chain
    }
    | summary_prompt
    | llm
    | parser
)

In [27]:
print("STEP 1: STRUCTURED EXTRACTION\n")
print(extract_chain.invoke({"notes": meeting_notes}))

STEP 1: STRUCTURED EXTRACTION

**1. Key Decisions:**
- The team agreed to prioritize critical interfaces before lower-value reports.

**2. Risks/Issues:**
- Delays in data migration are a concern for the client.
- Testing completion has slipped by one week due to environment issues.
- There is a risk that user training may start late unless materials are approved this week.

**3. Action Items:**
- Program manager to provide a revised milestone plan by Friday.
- Finance lead to clarify the view of budget burn and forecast.
- Approval of user training materials is needed this week to avoid delays.

**4. Deadlines:**
- Revised milestone plan due by Friday.
- User training materials need approval this week.
- Concise status summary for the executive sponsor due next Monday.

**5. Stakeholders Mentioned:**
- Client (general concern expressed)
- Program Manager
- Finance Lead
- Executive Sponsor
- Team (general agreement on prioritization)


In [28]:
print("\n" + "=" * 70 + "\n")

In [29]:
print("STEP 2: EXECUTIVE SUMMARY AND ACTIONS\n")
print(meeting_chain.invoke({"notes": meeting_notes}))

STEP 2: EXECUTIVE SUMMARY AND ACTIONS

### Executive Summary

In our recent meeting, the team made significant progress in aligning on project priorities and addressing potential risks. We have decided to focus on critical interfaces, ensuring that our efforts are directed towards high-impact deliverables. However, we face challenges with user training timelines and testing completion, which require immediate attention to mitigate any further delays. The upcoming week is crucial for finalizing key documents and clarifying budgetary concerns to maintain project momentum.

### Top Priorities This Week

1. **Approval of User Training Materials**: Ensure that training materials are reviewed and approved this week to avoid delays in user training.
2. **Revised Milestone Plan**: The program manager is tasked with providing a revised milestone plan by Friday to realign project timelines.
3. **Budget Clarification**: The finance lead must clarify the budget burn and forecast to ensure financia